In [ ]:
#main.py - import libraries
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [141]:
#main.py - read in data
test = pd.read_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/raw/test.csv")
train = pd.read_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/raw/train.csv")



In [142]:
#test info
test.head()
test.info()
test.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133776 entries, 0 to 133775
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         133776 non-null  int64  
 1   Age                133776 non-null  int64  
 2   Gender             133776 non-null  object 
 3   Tenure             106578 non-null  float64
 4   Usage Frequency    133776 non-null  int64  
 5   Support Calls      106238 non-null  object 
 6   Payment Delay      133776 non-null  int64  
 7   Subscription Type  133776 non-null  object 
 8   Contract Length    133776 non-null  object 
 9   Total Spend        133776 non-null  float64
 10  Last Interaction   106581 non-null  float64
 11  Last Due Date      133776 non-null  object 
 12  Last Payment Date  133776 non-null  object 
 13  Customer Status    133776 non-null  object 
dtypes: float64(3), int64(4), object(7)
memory usage: 14.3+ MB


Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Last Due Date',
       'Last Payment Date', 'Customer Status'],
      dtype='object')

In [143]:
#train info
train.head()
#train.info()
#train.columns


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Customer Status,Last Due Date,Last Payment Date,Churn
0,160831,34,Male,NaN,26,none,17.0,Basic,Quarterly,890.21,30.0,active,06-21,07-08,0
1,354435,30,Male,36.0,22,NaN,15.0,Basic,Annual,954.37,30.0,active,06-10,06-25,0
2,444576,23,Male,44.0,14,none,NaN,Premium,Annual,674.00,21.0,inactive,06-27,07-21,1
3,130847,24,Male,39.0,21,2,14.0,Standard,Quarterly,775.00,18.0,active,06-30,07-14,0
4,108121,26,Male,NaN,26,1,5.0,Basic,Quarterly,679.45,NaN,active,06-29,07-04,0


In [144]:
#preprocessing.py - preprocess functions

def fill_na_with_mode(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Fill missing values in the specified columns of the DataFrame with the mode."""
    for col in columns:
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)
    return df

def fill_cols_na_with_0s(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Fill missing values in the specified columns of the DataFrame with 0s."""
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

def one_hot_encode(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """One-hot encode specified categorical columns."""
    return pd.get_dummies(df, columns=columns, dtype = int)

def convert_data_types(df: pd.DataFrame, column_types: dict) -> pd.DataFrame:
    """Convert columns to specified data types."""
    for column, dtype in column_types.items():
        df[column] = df[column].astype(dtype)
    return df

def compute_payment_delay(df):
    df = df.copy()

    # Convert MM-DD → datetime (using fixed year)
    df['DueDate_dt'] = pd.to_datetime("2024-" + df['Last Due Date'], format="%Y-%m-%d", errors="coerce")
    df['PayDate_dt'] = pd.to_datetime("2024-" + df['Last Payment Date'], format="%Y-%m-%d", errors="coerce")

    # Calculate difference in days
    df['ComputedDelay'] = (df['PayDate_dt'] - df['DueDate_dt']).dt.days

    # Fill Payment Delay ONLY if it is NaN
    df['Payment Delay'] = df['Payment Delay'].fillna(df['ComputedDelay'])

    # Drop helper columns
    df = df.drop(columns=['DueDate_dt', 'PayDate_dt', 'ComputedDelay'])

    return df

In [145]:
#create copies of original data
train_clean = train.copy()
test_clean = test.copy()



In [146]:
train_clean.columns


Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Customer Status',
       'Last Due Date', 'Last Payment Date', 'Churn'],
      dtype='object')

In [147]:
#CLEAN DATA

#1-hot encode
train_clean = one_hot_encode(train_clean, ['Subscription Type', 'Contract Length'])
test_clean = one_hot_encode(test_clean, ['Subscription Type', 'Contract Length'])

#map binary columns
train_clean['Gender'] = train_clean['Gender'].map({'Male': 0, 'Female': 1})
test_clean['Gender'] = test_clean['Gender'].map({'Male': 0, 'Female': 1})

#remove customer status to prevent data leakage
train_clean = train_clean.drop(columns=['Customer Status'])
test_clean = test_clean.drop(columns=['Customer Status'])

#fill with mode
train_clean = fill_na_with_mode(train_clean, ['Tenure', 'Last Interaction'])
test_clean = fill_na_with_mode(test_clean, ['Tenure','Last Interaction'])

#fill Support Calls with zeros
train_clean = fill_cols_na_with_0s(train_clean, ['Support Calls'])
test_clean = fill_cols_na_with_0s(test_clean, ['Support Calls'])

#fill payment delay based on last due date and last payment date
train_clean = compute_payment_delay(train_clean)
test_clean = compute_payment_delay(test_clean)

#drop date columns (since we already have payment delay)
train_clean = train_clean.drop(columns=['Last Due Date', 'Last Payment Date'])
test_clean = test_clean.drop(columns=['Last Due Date', 'Last Payment Date'])

#train_clean['Spending Per Month'] = train_clean['Total Spend'] / train_clean['Tenure'].replace(0, 1)
#test_clean['Spending Per Month'] = test_clean['Total Spend'] / test_clean['Tenure'].replace(0, 1)

test_clean.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Subscription Type_Basic,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Annual,Contract Length_Monthly,Contract Length_Quarterly
0,262106,65,0,16.0,16,3.0,14,853.00,4.0,0,0,1,0,1,0
1,37061,23,0,55.0,12,3.0,6,860.00,21.0,0,1,0,1,0,0
2,361832,49,0,55.0,25,2.0,4,905.58,15.0,0,1,0,0,0,1
3,425516,25,1,48.0,11,7.0,16,446.00,8.0,0,1,0,0,0,1
4,60597,52,1,4.0,10,6.0,3,933.29,28.0,1,0,0,0,1,0


In [148]:
#create csvs of cleaned data
test_clean.to_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/processed/test_clean.csv", index=False)
train_clean.to_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/processed/train_clean.csv", index=False)

In [149]:
#train knn model
def train_knn_model(X_train: pd.DataFrame, y_train: pd.Series, ) -> KNeighborsClassifier:
    """Train and return a 3-NN classifier."""
    model = KNeighborsClassifier(n_neighbors=100)
    model.fit(X_train, y_train)
    return model

In [150]:
#split data
y = train_clean['Churn']
X = train_clean.drop(columns=['Churn', 'CustomerID'])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

In [151]:
knn_model = train_knn_model(X_train, y_train)
y_val_pred_knn = knn_model.predict(X_val)
val_prob_knn = knn_model.predict_proba(X_val)[:, 1]

In [152]:
y_test_pred_knn= knn_model.predict(test_clean.drop(columns=['CustomerID']))
test_prob_knn = knn_model.predict_proba(test_clean.drop(columns=['CustomerID']))[:, 1]

In [153]:
"""DONT FORGET TO UPDATE SUBMISSION FILE NAME"""
submission = pd.DataFrame({
    'CustomerID': test['CustomerID'], 
    'Churn': test_prob_knn
})
submission.to_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/processed/knn_predictions_17.csv", index=False)

In [154]:
assert all(submission['CustomerID'] == test['CustomerID']), "IDs are misaligned!"


In [155]:
submission['Churn'].value_counts()

Churn
0.71    2673
0.70    2656
0.72    2641
0.73    2570
0.69    2563
        ... 
0.94     121
0.95      71
0.00      69
0.96      26
0.97      14
Name: count, Length: 98, dtype: int64

In [156]:
train_clean.columns

Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction',
       'Churn', 'Subscription Type_Basic', 'Subscription Type_Premium',
       'Subscription Type_Standard', 'Contract Length_Annual',
       'Contract Length_Monthly', 'Contract Length_Quarterly'],
      dtype='object')